# Notebook 03 — Scoring Validation & Ablation
**Owner: Member 3**

Run the full pipeline on all test pairs and validate that flagged timestamps
correspond to visually obvious differences. Tune the deviation threshold.
Ablation: normalization on/off, DTW vs. naive matching.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pose_extraction import load_keypoints
from normalization import normalize
from alignment import dtw_align, warping_path_to_timestamps
from scoring import (
    compute_joint_errors, per_part_error_over_time,
    find_off_moments, overall_score
)
from feedback import generate_feedback

In [ ]:
DATA_DIR = Path('../data')
FPS = 15.0
THRESHOLD = 0.25

results = []
for phrase_dir in sorted(DATA_DIR.iterdir()):
    kp_dir = phrase_dir / 'keypoints'
    bench_path  = kp_dir / 'benchmark_kp.npy'
    learner_path = kp_dir / 'learner_kp.npy'
    if not bench_path.exists() or not learner_path.exists():
        continue

    bench_kp = load_keypoints(bench_path)
    user_kp  = load_keypoints(learner_path)

    bench_norm = normalize(bench_kp)
    user_norm  = normalize(user_kp)
    bench_al, user_al, path = dtw_align(bench_norm, user_norm)
    ts = warping_path_to_timestamps(path, FPS)

    joint_err = compute_joint_errors(bench_al, user_al)
    part_err  = per_part_error_over_time(joint_err)
    intervals = find_off_moments(part_err, THRESHOLD, FPS, 0.5, ts)
    score     = overall_score(part_err)

    results.append({
        'phrase':    phrase_dir.name,
        'score':     score,
        'n_intervals': len(intervals),
        'intervals': intervals,
        'feedback':  generate_feedback(intervals),
    })
    print(f'{phrase_dir.name}: score={score:.1f}, off_intervals={len(intervals)}')
    for line in generate_feedback(intervals):
        print(f'  {line}')
    print()

In [ ]:
# Ablation: effect of normalization
# TODO: run the pipeline without normalize() and compare interval counts
print('Ablation placeholder — implement after verifying the baseline above.')

In [ ]:
# Threshold sensitivity sweep
thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]

if results:
    r = results[0]
    bench_kp = load_keypoints(DATA_DIR / r['phrase'] / 'keypoints' / 'benchmark_kp.npy')
    user_kp  = load_keypoints(DATA_DIR / r['phrase'] / 'keypoints' / 'learner_kp.npy')
    bench_al, user_al, path = dtw_align(normalize(bench_kp), normalize(user_kp))
    ts = warping_path_to_timestamps(path, FPS)
    part_err = per_part_error_over_time(compute_joint_errors(bench_al, user_al))

    counts = [len(find_off_moments(part_err, th, FPS, 0.5, ts)) for th in thresholds]
    plt.plot(thresholds, counts, marker='o')
    plt.xlabel('Threshold')
    plt.ylabel('Number of off intervals')
    plt.title(f'Threshold sensitivity — {r["phrase"]}')
    plt.grid(True)
    plt.show()